In [ ]:
# --- BLOCK 1: SETUP & SAFETY ---

# 1. Mount Google Drive (To save my work permanently)
from google.colab import drive
import os

# This will ask for permission to access your Drive. Click "Connect".
drive.mount('/content/drive')

# 2. Create a folder for your project
# We are making a folder named 'AI_Research_Project' in your Drive.
project_path = '/content/drive/My Drive/AI_Research_Project'

if not os.path.exists(project_path):
    os.makedirs(project_path)
    print(f"Created new project folder at: {project_path}")
else:
    print(f"Project folder found at: {project_path}")

# 3. Install necessary AI libraries
# We need 'transformers' for the model, 'datasets' for data, and 'accelerate' for GPU optimization.
print("Installing libraries... this may take a minute.")
!pip install -q transformers datasets scikit-learn accelerate torch

# 4. Verify GPU is working
import torch
if torch.cuda.is_available():
    print(f"Success! GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU not detected. Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# --- BLOCK 2: LOAD DATA ---
from datasets import load_dataset
import pandas as pd

print("Downloading HC3 dataset...")
dataset = load_dataset("Hello-SimpleAI/HC3", "default", revision="refs/convert/parquet")

raw_df = pd.DataFrame(dataset['train'])
data_list = []

print("Formatting data for binary classification...")

# Extract Human (0) and AI (1) text
for index, row in raw_df.iterrows():
    if len(row['human_answers']) > 0:
        data_list.append({'text': row['human_answers'][0], 'label': 0})
    if len(row['chatgpt_answers']) > 0:
        data_list.append({'text': row['chatgpt_answers'][0], 'label': 1})

# Shuffle the data
df = pd.DataFrame(data_list)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("\n--- DATA REPORT ---")
print(f"Total Samples Ready: {len(df)}")

In [ ]:
# --- BLOCK 3: TOKENIZATION ---
from transformers import AutoTokenizer
from datasets import Dataset
from sklearn.model_selection import train_test_split

# 1. Load RoBERTa Tokenizer
checkpoint = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# 2. Define the conversion rule
def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

# 3. Split 80% for Training, 20% for Testing
print("Splitting data...")
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Convert to Hugging Face format
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# 4. Apply Tokenizer (Map)
print("Tokenizing data... (This may take 1-2 minutes)")
tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

print("Tokenization Complete!")

In [ ]:
# --- DEMO BLOCK: TOKENIZATION OUTPUT ---
print("\n--- Tokenization Complete ---")
print(f"Training Set Size: {len(tokenized_train)}")
print(f"Testing Set Size: {len(tokenized_test)}")
print("Sample Input IDs:", tokenized_train[0]['input_ids'][:10])

In [ ]:
# --- BLOCK 4 : STABLE MODEL TRAINING ---
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import torch

# 1. Load Model (RoBERTa)
checkpoint = "roberta-base"
print(f"Loading stable model: {checkpoint}...")
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# Move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# 2. Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions)
    }

# 3. Training Config (Optimized for 96k Dataset & Speed)
training_args = TrainingArguments(
    output_dir="/content/drive/My Drive/AI_Research_Project/results", # Saved to Drive!
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,             # CHANGED TO 1: 96k samples is huge, 1 epoch is enough!
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,                      # Uses GPU acceleration for speed
    report_to="none"
)

# 4. Start Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("Starting training... Please wait.")
trainer.train()

# 5. Save Final Model to Google Drive
save_path = "/content/drive/My Drive/AI_Research_Project/saved_model_roberta"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Success! Model securely saved to: {save_path}")

In [ ]:
# --- BLOCK 5 : STRESS TESTING ---
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import torch.nn.functional as F

# 1. Load the model from GOOGLE DRIVE
model_path = "/content/drive/My Drive/AI_Research_Project/saved_model_roberta"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Fetching model from: {model_path}...")

try:
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model.to(device)
    print("✅ Model loaded successfully from Google Drive!")
except Exception as e:
    print("❌ Error: Could not load model. Is your Google Drive mounted?")
    raise e

# 2. Define the Detection Engine
def detect_ai_text(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = F.softmax(logits, dim=-1)
    ai_prob = probs[0][1].item()

    # Threshold: > 50% = AI
    label = "AI-Generated 🤖" if ai_prob > 0.5 else "Human-Written ✍️"
    confidence = ai_prob if ai_prob > 0.5 else 1 - ai_prob

    print(f"\n------------------------------------------------")
    print(f"Input: {text[:75]}...")
    print(f"Prediction: {label} (Confidence: {confidence:.2%})")
    print(f"------------------------------------------------")

# 3. RUN THE STRESS TESTS

print("\n🚀 STARTING MODEL EVALUATION...\n")

# EX 1: The "Formal Human" Trap (Testing Formality Bias)
print("TEST 1: Formal Human Writing (Academic/Robotic tone)")
detect_ai_text("""
The aforementioned methodology necessitates a rigorous examination of the underlying variables.
Consequently, the implementation of the RoBERTa architecture was prioritized.
""")

# EX 2: The "Paraphrased AI" Attack (Testing Latent Features)
print("\nTEST 2: AI Text disguised with slang (Quillbot/Paraphraser Attack)")
detect_ai_text("""
So, photosynthesis is basically how plants make their own food, right?
It's kinda like they take sunlight, water, and CO2 and turn it into glucose.
""")

# EX 3: The "Mixed Text" (Testing Document-Level Averages)
print("\nTEST 3: Half Human / Half AI Text")
detect_ai_text("""
i tried to fix the code myself but it was confusing lol. i kept getting error 404.
However, upon restarting the system, the kernel successfully re-initialized without latency.
""")

In [ ]:
# --- BLOCK 6: ADVERSARIAL STRESS TEST ---
# I use the same detection function from Block 5.

print("--- STARTING ADVERSARIAL STRESS TEST ---\n")

# TEST CASE 1: The "Paraphrased" AI Attack
# This text was generated by ChatGPT, then rewritten to remove "AI words".
# It uses slang ("gonna", "kinda") but keeps the AI sentence structure.
paraphrased_ai = """
So, photosynthesis is basically how plants make their own food, right?
It's kinda like they take sunlight, water, and CO2 and turn it into glucose.
Oxygen acts as a byproduct, which is super important for us humans to breathe.
Without this process, life on Earth would essentially cease to exist.
"""

# TEST CASE 2: The "Formal Human" Trap
# This is written by a HUMAN (me), but it uses complex, robotic language.
# A bad model will think this is AI. A good model should see the "irregularity".
formal_human = """
The aforementioned methodology necessitates a rigorous examination of the underlying variables.
Consequently, the implementation of the RoBERTa architecture was prioritized to mitigate numerical instability.
It is imperative to note that while convergence was achieved, the validation loss exhibited minor fluctuations.
"""

# TEST CASE 3: The "Hidden AI" (Mixed)
# First half is Human (typos, bad grammar). Second half is ChatGPT (perfect grammar).
mixed_text = """
i tried to fix the code myself but it was confusing lol. i kept getting error 404 and
my computer froze. However, upon restarting the system, the kernel successfully re-initialized,
allowing the subsequent cells to execute without further interruption or latency issues.
"""

# Run the predictions
tests = [
    ("Paraphrased AI (The 'Quillbot' Attack)", paraphrased_ai),
    ("Formal Human (The 'False Positive' Trap)", formal_human),
    ("Mixed Text (Human start -> AI end)", mixed_text)
]

for name, text in tests:
    print(f"TEST: {name}")
    detect_ai_text(text) # Uses the function from Block 5

In [ ]:
# --- BLOCK 7: FORMAL METRICS & EVALUATION ---
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

print("Running formal evaluation on the 20% unseen test dataset...")
print("This will take about 1-2 minutes...\n")

# 1. Get predictions from the Trainer on the test dataset
predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# 2. Print the Classification Report (Precision, Recall, F1)
print("===================================================")
print("             CLASSIFICATION REPORT                 ")
print("===================================================")
target_names = ['Human (Class 0)', 'AI (Class 1)']
print(classification_report(labels, preds, target_names=target_names, digits=4))

# 3. Generate a visual Confusion Matrix
print("\nGenerating Confusion Matrix Plot...")
cm = confusion_matrix(labels, preds)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.ylabel('Actual True Label', fontsize=12, fontweight='bold')
plt.xlabel('Model Predicted Label', fontsize=12, fontweight='bold')
plt.title('Confusion Matrix: RoBERTa AI Detection', fontsize=14, pad=15)
plt.show()

In [ ]:
# --- BLOCK 8: LIVE DEMO ---

print("==================================================")
print("        LIVE AI DETECTION ENGINE READY       ")
print("==================================================")

custom_text = input("\nPlease paste the text you want to evaluate:\n> ")

print("\nAnalyzing contextual patterns...")
detect_ai_text(custom_text)